# IndexTTS-2.5 — Colab API / Live-Latency Test (vLLM-Omni)

This notebook serves IndexTTS-2.5 through vLLM-Omni's OpenAI-compatible `/v1/audio/speech` endpoint.

**Important:** current IndexTTS-2.5 serving can accept `stream=true`, but its audio response is non-chunked. This notebook therefore measures end-to-end latency, first received HTTP content, and concurrency without calling it true realtime PCM streaming.

Use a fresh Colab GPU runtime.


In [ ]:
# 1) GPU
!nvidia-smi


In [ ]:
# 2) Install vLLM-Omni IndexTTS support and source repos
!pip -q install -U "vllm-omni[indextts2]" requests soundfile "huggingface_hub[cli,hf_xet]"
!rm -rf /content/vllm-omni-src /content/index-tts
!git clone -q --depth 1 https://github.com/vllm-project/vllm-omni.git /content/vllm-omni-src
!git clone -q --depth 1 https://github.com/index-tts/index-tts.git /content/index-tts


In [ ]:
# 3) Download IndexTTS-2.5 weights into the upstream nested checkpoints layout
!hf download IndexTeam/IndexTTS-2.5 --local-dir /content/index-tts/checkpoints
MODEL_PATH = '/content/index-tts'
print('Model root:', MODEL_PATH)
!ls -lh /content/index-tts/checkpoints | head


In [ ]:
# 4) Upload a reference voice WAV/MP3
from google.colab import files
import os
uploaded = files.upload()
REF_AUDIO = os.path.join('/content', next(iter(uploaded.keys())))
print('Reference:', REF_AUDIO)


In [ ]:
# 5) Start vLLM-Omni IndexTTS-2.5 server
import subprocess, time, requests, os
PORT = 8092
LOG = '/content/index_vllm.log'
try:
    server.terminate()
except Exception:
    pass
logf = open(LOG, 'w')
server = subprocess.Popen(
    [
        'vllm', 'serve', MODEL_PATH,
        '--omni', '--trust-remote-code', '--port', str(PORT),
        '--deploy-config', '/content/vllm-omni-src/vllm_omni/deploy/indextts2_5.yaml',
    ],
    cwd='/content/vllm-omni-src',
    stdout=logf, stderr=subprocess.STDOUT,
)
print('Starting server...')
ready = False
for i in range(180):
    time.sleep(2)
    try:
        r = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=2)
        if r.ok:
            ready = True
            break
    except Exception:
        pass
    if server.poll() is not None:
        break
if not ready:
    print(open(LOG).read()[-10000:])
    raise RuntimeError('IndexTTS vLLM server failed to start')
print('Server ready:', f'http://127.0.0.1:{PORT}')


In [ ]:
# 6) Normal API request
import base64, mimetypes, requests, time
from IPython.display import Audio, display
raw = open(REF_AUDIO, 'rb').read()
mime = mimetypes.guess_type(REF_AUDIO)[0] or 'audio/wav'
ref_data = f'data:{mime};base64,' + base64.b64encode(raw).decode()
payload = {
    'model': MODEL_PATH,
    'input': 'こんにちは。今日はどんな一日でしたか？ゆっくり話してみてください。',
    'response_format': 'wav',
    'speed': 1.0,
    'ref_audio': ref_data,
    'extra_params': {
        'lang': 'ja',
        'text_normalization': True,
        'emo_vector': [0.1, 0, 0, 0, 0, 0, 0, 0.4],
        'emo_alpha': 0.6,
    },
}
t0 = time.perf_counter()
r = requests.post(f'http://127.0.0.1:{PORT}/v1/audio/speech', json=payload, timeout=300)
elapsed = time.perf_counter() - t0
r.raise_for_status()
OUT = '/content/index_api.wav'
open(OUT, 'wb').write(r.content)
print(f'End-to-end API latency: {elapsed:.3f}s; bytes={len(r.content):,}')
display(Audio(OUT))


In [ ]:
# 7) stream=true behavior test — measure honestly, do not treat as Qwen-style PCM streaming
payload_stream = dict(payload)
payload_stream['stream'] = True
t0 = time.perf_counter()
first = None
parts = []
with requests.post(f'http://127.0.0.1:{PORT}/v1/audio/speech', json=payload_stream, stream=True, timeout=300) as r:
    r.raise_for_status()
    for part in r.iter_content(chunk_size=4096):
        if not part:
            continue
        if first is None:
            first = time.perf_counter()
        parts.append(part)
done = time.perf_counter()
print(f'First received content: {first - t0:.3f}s')
print(f'Total time            : {done - t0:.3f}s')
print(f'HTTP chunks observed  : {len(parts)}')
print('IndexTTS-2.5 is currently classified as non-chunk streaming in vLLM-Omni.')


In [ ]:
# 8) Concurrent request benchmark
from concurrent.futures import ThreadPoolExecutor
import statistics
def one_request(i):
    p = dict(payload)
    p['input'] = f'これは同時実行テストの{i}番目の文章です。自然に読んでください。'
    t0 = time.perf_counter()
    r = requests.post(f'http://127.0.0.1:{PORT}/v1/audio/speech', json=p, timeout=300)
    r.raise_for_status()
    return time.perf_counter() - t0
CONCURRENCY = 4
with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
    times = list(ex.map(one_request, range(CONCURRENCY)))
print('Latencies:', [round(x, 3) for x in times])
print('Mean:', round(statistics.mean(times), 3), 's')
print('Max :', round(max(times), 3), 's')


In [ ]:
# 9) Server log / cleanup
print(open('/content/index_vllm.log').read()[-5000:])
# server.terminate()
